# 6. COMPAS: both sides were right

The recidivism case, on the real data ProPublica released.

In 2016 ProPublica reported that COMPAS was biased against Black defendants.
Northpointe replied that the score was equally accurate for both groups. Both
claims are true. They cannot both be fixed.

This notebook reproduces both, then asks whether it is solvable.

- Chouldechova (2017), `papers/chouldechova-2017-fair-prediction.pdf`, written
  directly about this case
- Kleinberg, Mullainathan & Raghavan (2016), `papers/kleinberg-2016-inherent-tradeoffs.pdf`
- Data: <https://github.com/propublica/compas-analysis>, `../data/`

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv('../data/compas-scores-two-years.csv')

# ProPublica's own filtering
df = df[(df.days_b_screening_arrest.abs() <= 30) & (df.is_recid != -1) &
        (df.c_charge_degree != 'O') & (df.score_text != 'N/A')]
df = df[df.race.isin(['African-American', 'Caucasian'])].copy()
df['high_risk'] = df.decile_score >= 5

black = df.race == 'African-American'
white = df.race == 'Caucasian'
print(f'{len(df)} defendants: {black.sum()} Black, {white.sum()} white')

5278 defendants: 3175 Black, 2103 white


## 1. The base rates genuinely differ

This is the fact that makes everything downstream unavoidable. It is also the
fact that is hardest to interpret, and section 5 comes back to it.

In [2]:
for name, mask in (('African-American', black), ('Caucasian', white)):
    print(f'  {name:<18} two-year recidivism = {df[mask].two_year_recid.mean():.1%}')

  African-American   two-year recidivism = 52.3%
  Caucasian          two-year recidivism = 39.1%


## 2. Northpointe's defence: the score is calibrated

A decile of 7 should mean the same probability of reoffending regardless of race.
If it does, the score is not lying to anyone.

In [3]:
print(f'  {"decile":<8}{"Black":>10}{"white":>10}{"gap":>8}')
for d in range(1, 11):
    s = df[df.decile_score == d]
    b = s[s.race == 'African-American'].two_year_recid.mean()
    w = s[s.race == 'Caucasian'].two_year_recid.mean()
    print(f'  {d:<8}{b:>10.0%}{w:>10.0%}{b - w:>+8.0%}')

  decile       Black     white     gap
  1              23%       21%     +2%
  2              30%       31%     -1%
  3              42%       34%     +7%
  4              47%       40%     +7%
  5              49%       46%     +3%
  6              59%       58%     +1%
  7              61%       60%     +1%
  8              71%       75%     -4%
  9              72%       71%     +1%
  10             84%       70%    +14%


Calibration holds well through decile 9. It breaks at decile 10, where the gap
is widest and the consequences are largest. That is worth noticing: the defence
is strongest exactly where the stakes are lowest.

## 3. ProPublica's finding: the errors are not shared equally

A false positive is a defendant labelled high risk who did **not** reoffend.

In [4]:
print(f'  {"":<18}{"FPR":>8}{"FNR":>8}')
for name, mask in (('African-American', black), ('Caucasian', white)):
    g = df[mask]
    fpr = g[g.two_year_recid == 0].high_risk.mean()
    fnr = 1 - g[g.two_year_recid == 1].high_risk.mean()
    print(f'  {name:<18}{fpr:>8.1%}{fnr:>8.1%}')

                         FPR     FNR
  African-American     42.3%   28.5%
  Caucasian            22.0%   49.6%


Nearly twice the false positive rate, and roughly half the false negative rate.
The errors do not cancel; they point in opposite directions for the two groups.

Both parties were reading the same file. Neither was misrepresenting it.

## 4. Can it be fixed?

Try it. Equalise the false positive rate by choosing a different threshold for
each group, then look at what that costs.

In [5]:
target_fpr = 0.30

def threshold_for_fpr(mask, target):
    g = df[mask]
    negatives = g[g.two_year_recid == 0].decile_score
    for t in range(1, 11):
        if (negatives >= t).mean() <= target:
            return t
    return 11


t_black = threshold_for_fpr(black, target_fpr)
t_white = threshold_for_fpr(white, target_fpr)
df['equalised'] = np.where(black, df.decile_score >= t_black, df.decile_score >= t_white)

print(f'thresholds chosen: Black >= {t_black},  white >= {t_white}\n')
print(f'  {"":<18}{"FPR":>8}{"FNR":>8}{"PPV":>8}{"flagged":>10}')
for name, mask in (('African-American', black), ('Caucasian', white)):
    g = df[mask]
    fpr = g[g.two_year_recid == 0].equalised.mean()
    fnr = 1 - g[g.two_year_recid == 1].equalised.mean()
    ppv = g[g.equalised].two_year_recid.mean()
    print(f'  {name:<18}{fpr:>8.1%}{fnr:>8.1%}{ppv:>8.1%}{g.equalised.mean():>10.1%}')

thresholds chosen: Black >= 7,  white >= 5

                         FPR     FNR     PPV   flagged
  African-American     22.8%   49.2%   71.0%     37.4%
  Caucasian            22.0%   49.6%   59.5%     33.1%


The false positive rates now match. Two things were spent to buy that.

The **positive predictive value** now differs between groups, so "flagged as high
risk" means something different depending on race. And the **thresholds
differ**, so two defendants with the same decile score get different decisions
because of their race. In a bail hearing that is not a technical detail.

This is Chouldechova's result, arrived at by hand: with unequal base rates you
choose between calibration, equal error rates, and equal treatment of equal
scores. You do not get all three.

## 5. The part that is not a maths problem

`two_year_recid` is **re-arrest**, not reoffending. The two differ by however
much policing intensity differs between neighbourhoods, and nobody in this
dataset knows by how much. So some unknown share of the 52.3 vs 39.1 gap in
section 1 is enforcement rather than conduct.

That matters more than any of the arithmetic above, because every fairness
criterion here is defined against that label. If the label is a measurement of
policing, then a perfectly calibrated model is a perfectly calibrated model of
policing, and calling its output "risk of reoffending" is a claim nobody has
established.

Second, the prediction is not passive. Detention costs employment and housing,
which raises the probability of the outcome being predicted. A model that
influences the thing it forecasts cannot be validated the way a weather forecast
can.

## Is it solvable?

Not as posed. What is available:

- **A better label.** Convictions rather than arrests is better and still not
  clean. This is the largest available improvement and it is a data collection
  problem, not a modelling one.
- **An explicit choice of which error to equalise,** argued in public, with the
  reasoning recorded. That is a policy act, and it is Blackstone's question,
  which predates all of this by three centuries.
- **Restricting features to individual conduct** rather than circumstance, at a
  known cost in accuracy.
- **Deciding the decision should not be automated.** A legitimate outcome, and
  the one this literature is least willing to state.

What the technical work can do is map the trade-off precisely and refuse to let
anyone pretend it is not there. It cannot choose the point on it.